In [1]:
import torch
import pandas as pd
from transformers import WhisperForConditionalGeneration, WhisperProcessor, GPT2LMHeadModel, GPT2Tokenizer, AutoModelForCausalLM, AutoTokenizer
import torchaudio
from tqdm import tqdm
import random

ckpt_path = '/home/aseems/Improving-ASR-with-LLM-Description/prompt_15epoch/results/test/checkpoint-31312'

whisper_model = WhisperForConditionalGeneration.from_pretrained(ckpt_path).to('cuda')
whisper_processor = WhisperProcessor.from_pretrained("openai/whisper-small")
df = pd.read_csv('/home/aseems/Improving-ASR-with-LLM-Description/data/blind_data_mod.csv')

/home/aseems/anaconda3/envs/pytorch/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [50]:
idx = random.randint(0, len(df))
input_path = df['path'][idx]
text = df['text'][idx]

audio, sr = torchaudio.load(input_path)
input_ids = whisper_processor(audio[0], sampling_rate=16000, return_tensors='pt').input_features.to('cuda')

num_beams = 5
with torch.no_grad():
    candidates = whisper_model.generate(
        input_ids,
        num_beams=num_beams,
        num_return_sequences=num_beams,
        temperature=0.8,
        # early_stopping=True,
    )
decoded_candidates = whisper_processor.batch_decode(candidates, skip_special_tokens=True)

print(decoded_candidates)
# print(text)

['अपने सिग्नल के लिए एक इनपुट चुनते हैं', 'अपने सिग्नल के लिए एक इनको चुनते हैं', 'अपने सिग्नल के लिए एक इनपुट चुनते हैं', 'अपने सिग्नल के लिए एक इनको चुनते हैं', 'अपने सिग्नल के लिए एक इनपुट चुनते हैं']


In [51]:
text

'अपने signal के लिए एक input चुनते हैं'

In [16]:
text

'mark up fact factorial symbol दर्शाता है'

In [13]:
decoded_candidates

['जैसा का परिणाम अब jemail नहीं है इसकी जगह पर सबसे ऊपर परिणाम या हूँ mail है',
 'जैसा का परिणाम अब जीमेल नहीं है इसकी जगह पर सबसे ऊपर परिणाम या हूँ मेल है',
 'जैसा का परिणाम अब jemail नहीं है इसकी जगह पर सबसे ऊपर परिणाम या हूँ mail है',
 'जैसा का परिणाम अब jemail नहीं है इसकी जगह पर सबसे ऊपर परिणाम या हूँ mail है',
 'cursor का परराम अब jemail नहीं है इसकी जगह पर सबसे उपरप्रकाम या हूँ mail है']

In [9]:
decoded_candidates

['यह मैंने search enginelist नामक एक dialog box खुलता है',
 'यह मैंने search indian lists नामक एक dialog box खोलता है',
 'यह मैंने search engine list नामक एक dialog box खुलता है',
 'यह मैंने search enginelist नामक एक dialog box खोलता है',
 'यह मैंने search engine list नामक एक dialog box खुलता है']

In [5]:
import pandas as pd

with open('csv_pred_ref/zeroshot_predictions.txt', 'r') as f:
    zero_shot = f.readlines()

with open('csv_pred_ref/prompted_references_31312.txt', 'r') as f:
    ground_truth = f.readlines()

with open('csv_pred_ref/finetuned_predictions_16480.txt', 'r') as f:
    finetuned = f.readlines()

with open('csv_pred_ref/prompted_predictions_31312.txt', 'r') as f:
    prompted = f.readlines()

# with open('gpt2_predictions_31312.txt', 'r') as f:
#     gpt2 = f.readlines()
gpt2 = pd.read_csv('/home/aseems/Improving-ASR-with-LLM-Description/csv_pred_ref/rescored_output_beam_5.csv')['best_candidate'].tolist()

with open('csv_pred_ref/llama3_1_8b_predictions_beam_5.txt', 'r') as f:
    llama31 = f.readlines()

llama32 = pd.read_csv('/home/aseems/Improving-ASR-with-LLM-Description/csv_pred_ref/llama_3.2_rescored_output_beam_5.csv')['best_candidate'].tolist()



prompted = [x.strip() for x in prompted]
ground_truth = [x.strip() for x in ground_truth]
finetuned = [x.strip() for x in finetuned]
zero_shot = [x.strip() for x in zero_shot]
llama31 = [x.strip() for x in llama31]
gpt2 = [x.strip() for x in gpt2]
llama32 = [x.strip() for x in llama32]


ans = []
for gt, zero, fin, prom, gpt, l31, l32 in zip(ground_truth, zero_shot, finetuned, prompted, gpt2, llama31, llama32):
    # if prom == gt and prom != fin:
    #     ans.append((gt, zero, fin, prom, gpt, l31, l32))

    # if prom != gt and fin!=gt and gt == l31:
    #     ans.append((gt, zero, fin, prom, gpt, l31, l32))

    if prom != gt and fin!=gt and gt != l31:
        ans.append((gt, zero, fin, prom, gpt, l31, l32))

In [6]:
import random
idx = random.randint(0, len(ans))
ref = ans[idx][0]
zero = ans[idx][1]
fin = ans[idx][2]
prom = ans[idx][3]
gpt2 = ans[idx][4]
l31 = ans[idx][5]
l32 = ans[idx][6]

print(f'Ground Truth: {ref}')
print(f'Zero Shot: {zero}')
print(f'Fine Tuned: {fin}')
print(f'Prompted: {prom}')
print(f'GPT2: {gpt2}')
print(f'llama3.1: {l31}')
print(f'llama 3.2: {l32}')

Ground Truth: 1123 put insulin . fasta file के लिए contents दिखाता है
Zero Shot: 1123  अप्रोट इंसुलिन ड़ प्रश्टा फाँँईल के लिए, ख़न्टेंस दिखाता है.
Fine Tuned: 1123 आउटपुट insulin dot first फाइल के लिए कंटेंट्स दिखाता है
Prompted: 1123 default insulin dot firster file के लिए कंटेंट्स दिखाता है
GPT2: default installation dot firster file के लिए कंटेंट्स दिखाता है
llama3.1: 1123 default installation dot firster file के लिए कंटेंट्स दिखाता है
llama 3.2: default installation dot firster file के लिए कंटेंट्स दिखाता है


In [78]:
#
#194
#57 -> best till now
#23
#73
#9
#132
#172
#166
#18
#135
#164 -> best till now
#156
#61 -> homophones problem solving
#183
#144
idx

144

In [1]:
import pandas as pd

df = pd.read_csv('/home/aseems/Improving-ASR-with-LLM-Description/data/blind_data_mod.csv')
df['audio'] = df['path']
df.to_csv('/home/aseems/Improving-ASR-with-LLM-Description/data/blind_data_for_prompting_whisper.csv', index=False)

In [5]:
## change the directory path
test = pd.read_csv('/home/aseems/Improving-ASR-with-LLM-Description/data/blind_data_mod.csv')
test['path'] = test['path'].str.replace('/raid/home/shada/data', '/home/aseems/proposed_method')
test.to_csv('/home/aseems/Improving-ASR-with-LLM-Description/data/blind_data_mod.csv', index=False)

In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '4'

from transformers_prompt import Seq2SeqTrainingArguments, Seq2SeqTrainer, WhisperPromptForConditionalGeneration, GenerationConfig, WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor
import torch
from utils_prompt import compute_wer, DataCollatorSpeechS2SWhitPadding
from data.dataloader import CodeMixedWhisperDataset

feature_extractor = WhisperFeatureExtractor.from_pretrained('openai/whisper-small')
tokenizer = WhisperTokenizer.from_pretrained('openai/whisper-small', language='hi', task='transcribe')
processor = WhisperProcessor.from_pretrained('openai/whisper-small', language='hi', task='transcribe')


data_train = CodeMixedWhisperDataset(csv_path='/raid/home/shada/Improving-ASR-with-LLM-Description/data/train_output_with_context_mod.csv', feature_extractor=feature_extractor, tokenizer=tokenizer, prompt=True) 
data_eval = CodeMixedWhisperDataset(csv_path='/raid/home/shada/Improving-ASR-with-LLM-Description/data/test_output_with_context_deleted_entries.csv', feature_extractor=feature_extractor, tokenizer=tokenizer, prompt=True) 

data_collator = DataCollatorSpeechS2SWhitPadding(processor=processor)
train_dl = torch.utils.data.DataLoader(data_train, batch_size=16, shuffle=True, collate_fn=data_collator)
# eval_dl = torch.utils.data.DataLoader(data_eval, batch_size=1, shuffle=False, collate_fn=data_collator)
# model = WhisperPromptForConditionalGeneration.from_pretrained(f'openai/whisper-small').to('cuda')

/raid/home/shada/anaconda3/envs/llm-description/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/raid/home/shada/anaconda3/envs/llm-description/lib/python3.9/site-packages/transformers/utils/import_utils.py:519: FutureWarning: `is_torch_tpu_available` is deprecated and will be removed in 4.41.0. Please use the `is_torch_xla_available` instead.
  warnings.warn(
/raid/home/shada/anaconda3/envs/llm-description/lib/python3.9/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens

In [2]:
# from tqdm import tqdm

# for row in tqdm(data_train):
#     if len(row['labels']) > 448:
#         print(row['input_features'])
#         print(row['labels'])
#         break

In [2]:
len(data_train)

12748

In [2]:
from tqdm import tqdm

shapes = []
corrupted_labels = []
for batch in tqdm(train_dl):
    if batch['labels'].shape[1] > 448:
        print(batch['prompts'].shape)
        print(batch['labels'].shape)
        print(batch['labels'])
        corrupted_labels.append(batch['labels'])
        break
    else:
        shapes.append(batch['labels'].shape[1])

100%|██████████| 797/797 [03:13<00:00,  4.12it/s]


## Creating csv for blind data

In [4]:
import torchaudio
x, sr = torchaudio.load('/home/aseems/GV_Train_100h/Audio/13-00399-05.mp3')
x

tensor([[-0.0002, -0.0005, -0.0003,  ...,  0.0001, -0.0004, -0.0002]])

In [6]:
import pandas as pd
import os

df = pd.DataFrame(columns=['path', 'text'])

dir_path = '/raid/home/shada/data/blind_data'
# need to iterate in this directory, and create a dataframe with path and text, text is the name of the file.txt
for root, dirs, files in os.walk(dir_path):
    for file in files:
        if file.endswith('.txt'):
            with open(os.path.join(root, file), 'r') as f:
                text = f.read()
                # path is a audio path for respective text, replace .txt with .wav

                df = pd.concat([df, pd.DataFrame({'path': os.path.join(root, file).replace('.txt', '.wav'), 'text': [text]})], ignore_index=True)

## Evaluation Prompting Whisper on Blind Data

In [6]:
import pandas as pd

cnt = 0
ans = []
df = pd.read_csv('/raid/home/shada/Improving-ASR-with-LLM-Description/data/blind_data.csv')
for row in df.iterrows():
    if type(row[1]['text']) is float:
        # delete this row
        df.drop(row[0], inplace=True)
        cnt += 1


df.to_csv('/raid/home/shada/Improving-ASR-with-LLM-Description/data/blind_data_mod.csv', index=False)

## Inferencing from the Whisper

In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
from transformers import WhisperForConditionalGeneration, WhisperProcessor

processor = WhisperProcessor.from_pretrained('openai/whisper-small', language='hi', task='transcribe')
model = WhisperForConditionalGeneration.from_pretrained('/home/aseems/Improving-ASR-with-LLM-Description/prompt_15epoch/results/test/checkpoint-4944').to('cuda')

/home/aseems/anaconda3/envs/pytorch/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torchaudio
import pandas as pd
import random
import tqdm
import torch    

df = pd.read_csv('/home/aseems/Improving-ASR-with-LLM-Description/data/blind_data_mod.csv')

def generate_predictions(df):
    pre = []
    ref = []
    for idx in tqdm.tqdm(range(len(df))):
        audio, sr = torchaudio.load(df.iloc[idx]['path'])
        input_ids = processor(audio[0], sampling_rate=16000, return_tensors='pt').input_features.to('cuda')
        with torch.no_grad():
            predicted_ids = model.generate(input_ids)
            transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)
            ref.append(df.iloc[idx]['text'])
            pre.append(transcription[0])

    return pre, ref


def save_predictions(pre, ref):
    idx = 0 
    with open("./deepseek_references_beam_5.txt", "w") as output:
        # iterate in list and store value in each line
        for row in ref:
            output.write(str(idx)+' '+str(row) + '\n')
            idx += 1

    idx = 0
    #save list as a text file
    with open("./deepseek_predictions_beam_5.txt", "w") as output:
        # iterate in list and store value in each line
        for row in pre:
            # create a string with the row value
            output.write(str(idx)+' '+str(row) + '\n')
            idx += 1


In [4]:
import evaluate
wer = evaluate.load('wer')
# wer.compute(predictions=my_preds, references=my_refs)

0.28902520726907294

In [ ]:
my_preds, my_refs = generate_predictions(df)
# save_predictions(my_preds, my_refs)

  0%|          | 0/4026 [00:00<?, ?it/s]

Due to a bug fix in https://github.com/huggingface/transformers/pull/28687 transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English.This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`.
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.43.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
  0%|          | 6/4026 [00:34<4:28:58,  4.01s/it] 

In [ ]:
# evaluate model on the single example.
import torchaudio
import pandas as pd
import random
import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from IPython.display import Audio

df = pd.read_csv('/raid/home/shada/Improving-ASR-with-LLM-Description/data/blind_data_mod.csv')


processor = WhisperProcessor.from_pretrained('openai/whisper-small', language='hi', task='transcribe')
model = WhisperForConditionalGeneration.from_pretrained('./results_15epoch/results/test/checkpoint-31312').to('cuda')

idx = random.randint(0, len(df))
audio, sr = torchaudio.load(df.iloc[idx]['path'])
input_ids = processor(audio[0], sampling_rate=16000, return_tensors='pt').input_features.to('cuda')
with torch.no_grad():
    finetuned_predicted_ids = model.generate(input_ids, num_beams=5, num_return_sequences=5, output_scores=True, return_dict_in_generate=True)
    finetuned_transcription = processor.batch_decode(finetuned_predicted_ids.sequences, skip_special_tokens=True) 


print(f'Finetuned Transcription: {finetuned_transcription}') 
print(f'Reference: {df.iloc[idx]["text"]}')
Audio(audio[0], rate=16000)
print(finetuned_predicted_ids.sequences_scores)


/raid/home/shada/anaconda3/envs/llm-description/lib/python3.9/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


AttributeError: 'NoneType' object has no attribute 'shape'

In [ ]:
# finetuned_predicted_ids = model.generate(input_ids, num_beams=1, num_return_sequences=1)
# processor.batch_decode(finetuned_predicted_ids, skip_special_tokens=True)

['सभी वेरिएबल डिक्लेरिशन सेमीकॉलम के साथ समाप्त होने चाहिए']

In [ ]:
# # join the csv files
# import pandas as pd

# csv1 = pd.read_csv('/home/aseems/Improving-ASR-with-LLM-Description/rescored_new_gpt4_beam_5_2.csv')
# csv2 = pd.read_csv('/home/aseems/Improving-ASR-with-LLM-Description/rescored_new_gpt4_beam_5_2_1.csv')
# csv3 = pd.read_csv('/home/aseems/Improving-ASR-with-LLM-Description/rescored_new_gpt4_beam_5_2_2.csv')
# # now join the csv files
# csv = pd.concat([csv1, csv2, csv3], ignore_index=True)
# csv = csv.drop_duplicates()

# csv.to_csv('rescored_gpt4_o_mini.csv', index=False)

In [4]:
import pandas as pd
import evaluate

df = pd.read_csv('/home/aseems/Improving-ASR-with-LLM-Description/rescored_gpt4_o_mini.csv')
ref = []
pred = []
for row in df.iterrows():
    ref.append(row[1]['ref_text'])
    pred.append(row[1]['best_candidate'])

metrics = evaluate.load('wer')
print(metrics.compute(predictions=pred, references=ref))


def save_predictions(pre, ref):
    idx = 0 
    with open("./gpt4_o_mini_references_beam_5.txt", "w") as output:
        # iterate in list and store value in each line
        for row in ref:
            output.write(str(idx)+' '+str(row) + '\n')
            idx += 1

    idx = 0
    #save list as a text file
    with open("./gpt4_o_min_predictions_beam_5.txt", "w") as output:
        # iterate in list and store value in each line
        for row in pre:
            # create a string with the row value
            output.write(str(idx)+' '+str(row) + '\n')
            idx += 1

/home/aseems/anaconda3/envs/pytorch/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


0.2737811554067109


In [5]:
save_predictions(pred, ref)

## Evaluate the whisper small model without prompt.

In [24]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from datasets import load_dataset, Audio
import evaluate
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import numpy as np
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
import torchaudio


# Function to test Whisper with prompts
def test_whisper_with_prompts(audio_path, model_name="openai/whisper-small"):
    # Load model and processor
    processor = WhisperProcessor.from_pretrained('openai/whisper-small',task="transcribe")
    model = WhisperForConditionalGeneration.from_pretrained(model_name,)

    audio, sr = torchaudio.load(audio_path)
    
    # Load and process audio
    audio_input = processor(
        audio[0],   
        return_tensors="pt",
        sampling_rate=16000
    )
    
    # Generate transcription with prompt
    generated_ids = model.generate(
        audio_input.input_features,
    )
    
    transcription = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return transcription

# Function to test Whisper with prompts
test_whisper_with_prompts('/raid/home/shada/data/blind_data/543096_UwcQJEuG2r7AD8vq_0035.wav', model_name="openai/whisper-small")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


'जब यह रिस्टार्ट होता है नई थीम्स लागू होती हैं'

## Evaluating the whisper small model with few shot prompt.

In [22]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from datasets import load_dataset, Audio
import evaluate
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import numpy as np
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
import torchaudio



# Function to prepare few-shot prompts
def prepare_prompts():
    return [
        {
            "audio": "[Technical explanation about programming]",
            "text": "हमें database में tables को join करने के लिए SQL queries का use करना होगा"
        },
        # {
        #     # "audio": "[Programming concept explanation]",
        #     # "text": "array को sort करने के लिए quicksort algorithm का time complexity O(n log n) होता है"
        # }
    ]

# Function to format prompts for Whisper
def format_prompt(prompts):
    formatted = ""
    for p in prompts:
        formatted += f"{p['text']}\n"
    return formatted.strip()


# Function to test Whisper with prompts
def test_whisper_with_prompts(audio_path, model_name="openai/whisper-small"):
    # Load model and processor
    processor = WhisperProcessor.from_pretrained('openai/whisper-small', task="transcribe")
    model = WhisperForConditionalGeneration.from_pretrained(model_name)
    
    # Prepare prompts
    prompts = prepare_prompts()
    formatted_prompt = format_prompt(prompts)
    prompt_ids = processor.get_prompt_ids(formatted_prompt, return_tensors="pt").to(model.device)

    audio, sr = torchaudio.load(audio_path)
    
    # Load and process audio
    audio_input = processor(
        audio[0],   
        return_tensors="pt",
        sampling_rate=16000
    )
    
    # Generate transcription with prompt
    forced_decoder_ids = processor.get_decoder_prompt_ids(language="hi", task="transcribe")
    generated_ids = model.generate(
        audio_input.input_features,
        # forced_decoder_ids=forced_decoder_ids,
        prompt_ids=prompt_ids,
    )
    
    transcription = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    transcription = transcription.replace(" "+formatted_prompt, "", 1)
    return transcription


# Function to test Whisper with prompts
test_whisper_with_prompts('/raid/home/shada/data/blind_data/543096_UwcQJEuG2r7AD8vq_0035.wav', model_name="openai/whisper-small")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


' जब यहर यहर यह चाथ होता है नहीं फीम्स लागु होती हैं'

## Evaluating the finetuned whisper small model with few shot prompt.

In [29]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from datasets import load_dataset, Audio
import evaluate
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import numpy as np
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
import torchaudio



# Function to prepare few-shot prompts
def prepare_prompts():
    return [
        {
            "audio": "[Technical explanation about programming]",
            "text": "code-switching text which contains mix of devanagari and english script. example is यह एक example है"
        },
        # {
        #     "audio": "[Programming concept explanation]",
        #     "text": "example: array को sort करने के लिए quicksort algorithm का time complexity O(n log n) होता है"
        # }
    ]

# Function to format prompts for Whisper
def format_prompt(prompts):
    formatted = ""
    for p in prompts:
        formatted += f"{p['text']}\n"
    return formatted.strip()


# Function to test Whisper with prompts
def test_whisper_with_prompts(audio_path, model_name="openai/whisper-small"):
    # Load model and processor
    processor = WhisperProcessor.from_pretrained('openai/whisper-small', task="transcribe")
    model = WhisperForConditionalGeneration.from_pretrained(model_name)
    
    # Prepare prompts
    prompts = prepare_prompts()
    formatted_prompt = format_prompt(prompts)
    prompt_ids = processor.get_prompt_ids(formatted_prompt, return_tensors="pt").to(model.device)

    audio, sr = torchaudio.load(audio_path)
    
    # Load and process audio
    audio_input = processor(
        audio[0],   
        return_tensors="pt",
        sampling_rate=16000
    )
    
    # Generate transcription with prompt
    forced_decoder_ids = processor.get_decoder_prompt_ids(language="hi", task="transcribe")
    generated_ids = model.generate(
        audio_input.input_features,
        # forced_decoder_ids=forced_decoder_ids,
        prompt_ids=prompt_ids,
    )
    
    transcription = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    transcription = transcription.replace(" "+formatted_prompt, "", 1)
    return transcription


# Function to test Whisper with prompts
test_whisper_with_prompts('/raid/home/shada/data/blind_data/543096_UwcQJEuG2r7AD8vq_0035.wav', model_name="./finetuning_15epoch/checkpoint-47792")

/raid/home/shada/anaconda3/envs/llm-description/lib/python3.9/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


'जजजबू बंइसंबंग्र करेंगर'

In [7]:
# Convert .mp3 file .wav file


import os
from pydub import AudioSegment

# Set paths
audio_dir = "/home/aseems/GV_Eval_3h/Audio"
output_dir = "/home/aseems/GV_Eval_3h/Audio_wav"
os.makedirs(output_dir, exist_ok=True)

# Convert all MP3 files to WAV
for file in os.listdir(audio_dir):
    if file.endswith(".mp3"):
        mp3_path = os.path.join(audio_dir, file)
        wav_path = os.path.join(output_dir, file.replace(".mp3", ".wav"))
        
        audio = AudioSegment.from_mp3(mp3_path)
        audio.export(wav_path, format="wav")

print("Conversion completed!")


Conversion completed!
